In [1]:
print("gj")

gj


Wrapper Feature Selection using classical evolutionary algorithms

In [2]:
from mealpy import *
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
ALPHA = 1
BETA = 0.001

# Objective Function: Wrapper Feature Selection
def feature_selection_fitness(solution, train_x_sc, y_train, test_x_sc, y_test):
    # Decode binary solution
    selected_features = np.where(solution > 0.5)[0]
    num_feat = len(selected_features)
    if num_feat == 0:  # Avoid empty feature set
        return 1e6  # Penalize empty selection
    
    # Train model
    model = RandomForestClassifier(random_state=42)
    model.fit(train_x_sc[:, selected_features], y_train)

    # Evaluate model
    # ac_scores = cross_val_score(model, test_x_sc[:, selected_features], y_test, cv=5, scoring='accuracy')
    # error_rate = 1 - ac_scores.mean()
    
    y_pred = model.predict(test_x_sc[:, selected_features])
    accuracy = accuracy_score(y_test, y_pred)
    error_rate = 1 - accuracy

    fit = ALPHA * error_rate + BETA * num_feat
    
    #fitness function = alpha * error rate  + beta * number of features
    return fit #accuracy#/(num_feat+0.00001)  # Max. accuracy

# Wrapper Optimization
def wrapper_feature_selection_evo(train_x_sc, y_train, test_x_sc, y_test, algo):
    n_features = train_x_sc.shape[1]
    problem_dict = {
        "obj_func": lambda solution: feature_selection_fitness(solution, train_x_sc, y_train, test_x_sc, y_test),
        "minmax": "min",         # Minimize the fitness function
        "bounds": FloatVar(lb=(0,) * n_features, ub=(1,) * n_features, name="delta"),
    }
    g_best = algo.solve(problem_dict)
    return g_best


def get_best_features_evo(algo, train_x_sc, y_train, test_x_sc, y_test):
    
    print("Algo: ",algo)
    g_best = wrapper_feature_selection_evo(train_x_sc, y_train, test_x_sc, y_test, algo)
    # print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")

    b_selected_features = np.where(g_best.solution > 0.5)[0]
    print("b_selected_features: ",b_selected_features)

    model = RandomForestClassifier(random_state=42)
    model.fit(train_x_sc[:, b_selected_features], y_train)
    #Evaluate model
    y_pred = model.predict(test_x_sc[:, b_selected_features])
    accuracy = accuracy_score(y_test, y_pred)
    print("accuracy: ",accuracy)

    return b_selected_features, accuracy

In [4]:
# model = RandomForestClassifier(random_state=42)
# algo = PSO.AIW_PSO(epoch=10, pop_size=30)
# b_selected_features, accuracy = get_best_features_evo(algo, model, train_x_sc, y_train, test_x_sc, y_test)

In [5]:
def get_data(f_p, test_size_p = 0.3, random_state_p = 42):
    print("dataset: ",f_p)
    csv = pd.read_csv(f_p)
    
    target = csv.columns[-1]
    X = csv.drop([target], axis=1)
    y = csv[target]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size_p, random_state=random_state_p)

    scaler = StandardScaler()
    train_x_sc = scaler.fit_transform(X_train)
    test_x_sc = scaler.transform(X_test)

    return {'train_x_sc': train_x_sc, 'y_train': y_train, 'test_x_sc':test_x_sc, 'y_test':y_test}

Particle Swarm Optimization

In [6]:
# print("Balanced datasets above dimension 15: ====== ")

# dir = "dim_above_15/balanced"

# bal_data_res = []

# for file in os.listdir(dir):
#     print("dataset: ",file)
#     f_p = os.path.join(dir, file)
#     csv = pd.read_csv(f_p)
    
#     target = csv.columns[-1]
#     X = csv.drop([target], axis=1)
#     y = csv[target]
    
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
#     train_df = pd.concat([X_train, y_train], axis=1)
#     test_df = pd.concat([X_test, y_test], axis=1)

#     scaler = StandardScaler()
#     train_x_sc = scaler.fit_transform(X_train)
#     test_x_sc = scaler.transform(X_test)

#     algo = PSO.AIW_PSO(epoch=10, pop_size=30)
#     b_selected_features, accuracy = get_best_features_evo(algo, train_x_sc, y_train, test_x_sc, y_test)

#     bal_data_res.append([file, b_selected_features, accuracy])

In [7]:
# print("Imbalanced datasets above dimension 15: ====== ")

# dir = "dim_above_15/imbalanced"

# imbal_data_res = []

# for file in os.listdir(dir):
#     print("dataset: ",file)
#     f_p = os.path.join(dir, file)
#     csv = pd.read_csv(f_p)
    
#     target = csv.columns[-1]
#     X = csv.drop([target], axis=1)
#     y = csv[target]
    
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
#     train_df = pd.concat([X_train, y_train], axis=1)
#     test_df = pd.concat([X_test, y_test], axis=1)

#     scaler = StandardScaler()
#     train_x_sc = scaler.fit_transform(X_train)
#     test_x_sc = scaler.transform(X_test)

#     algo = PSO.AIW_PSO(epoch=10, pop_size=30)
#     b_selected_features, accuracy = get_best_features_evo(algo, train_x_sc, y_train, test_x_sc, y_test)

#     bal_data_res.append([file, b_selected_features, accuracy])

In [8]:
iterations = 10
#pop_size = 30
pop_size = 5

evo_algorithms = {
    'Particle Swarm Optimization': PSO.AIW_PSO(epoch=iterations, pop_size=pop_size),
    'Manta Ray Forging Optimization': MRFO.OriginalMRFO(epoch=iterations, pop_size=pop_size),
    'Artificial Bee Colony (ABC)': ABC.OriginalABC(epoch=iterations, pop_size=pop_size),
    'Ant Colony Optimization Continous': ACOR.OriginalACOR(epoch=iterations, pop_size=pop_size),
    'Ant Lion Optimizer (ALO)': ALO.DevALO(epoch=iterations, pop_size=pop_size),
    'Adaptive Bat-inspired Algorithm (ABA)': BA.AdaptiveBA(epoch=iterations, pop_size=pop_size),
    'Bald Eagle Search (BES)': BES.OriginalBES(epoch=iterations, pop_size=10), #pop_size),
    'Cuckoo Search Algorithm (CSA)': CSA.OriginalCSA(epoch=iterations, pop_size=pop_size),
    'Cat Swarm Optimization (CSO)': CSO.OriginalCSO(epoch=iterations, pop_size=pop_size),
    'Dwarf Mongoose Optimization Algorithm (DMOA)': DMOA.DevDMOA(epoch=iterations, pop_size= 10), #pop_size),
    'Dragonfly Optimization (DO)': DO.OriginalDO(epoch=iterations, pop_size=pop_size),
    'Elephant Herding Optimization (EHO)': EHO.OriginalEHO(epoch=iterations, pop_size= 25) , #pop_size),
    'Firefly Algorithm (FFA)': FFA.OriginalFFA(epoch=iterations, pop_size=pop_size),
    'Grasshopper Optimization Algorithm (GOA)': GOA.OriginalGOA(epoch=iterations, pop_size=pop_size),
    'Honey Badger Algorithm (HBA)': HBA.OriginalHBA(epoch=iterations, pop_size=pop_size),
    'Sparrow Search Algorithm (SSA)': SSA.DevSSA(epoch=iterations, pop_size=pop_size),
}

for ev_k in evo_algorithms:
    print(evo_algorithms[ev_k])

AIW_PSO(epoch=10, pop_size=5, c1=2.05, c2=2.05, alpha=0.4)
OriginalMRFO(epoch=10, pop_size=5, somersault_range=2.0)
OriginalABC(epoch=10, pop_size=5, n_limits=25)
OriginalACOR(epoch=10, pop_size=5, sample_count=25, intent_factor=0.5, zeta=1.0)
DevALO(epoch=10, pop_size=5)
AdaptiveBA(epoch=10, pop_size=5, loudness_min=1.0, loudness_max=2.0, pr_min=0.15, pr_max=0.85, pf_min=-10.0, pf_max=10.0)
OriginalBES(epoch=10, pop_size=10, a_factor=10, R_factor=1.5, alpha=2.0, c1=2.0, c2=2.0)
OriginalCSA(epoch=10, pop_size=5, p_a=0.3)
OriginalCSO(epoch=10, pop_size=5, mixture_ratio=0.15, smp=5, spc=False, cdc=0.8, srd=0.15, c1=0.4, w_min=0.5, w_max=0.9, selected_strategy=1)
DevDMOA(epoch=10, pop_size=10, peep=2.0)
OriginalDO(epoch=10, pop_size=5)
OriginalEHO(epoch=10, pop_size=25, alpha=0.5, beta=0.5, n_clans=5)
OriginalFFA(epoch=10, pop_size=5, gamma=0.001, beta_base=2.0, alpha=0.2, alpha_damp=0.99, delta=0.05, exponent=2)
OriginalGOA(epoch=10, pop_size=5, c_min=4e-05, c_max=2.0)
OriginalHBA(epoch=

In [9]:
bal_dir = "dim_above_15/balanced"
imbal_dir = "dim_above_15/imbalanced"

done_list = os.listdir("DEAP_TRAD_FEAT_SEL/res_pop_5/above_15_res/")

for ev_k in evo_algorithms:
    
    try:
        if(ev_k+".csv" in done_list):
            continue
        
        algo = evo_algorithms[ev_k]
        bal_data_res = []
        for file in os.listdir(bal_dir):
            print("dataset: ",file)
            f_p = os.path.join(bal_dir, file)
            data = get_data(f_p=f_p)
            train_x_sc = data['train_x_sc']
            test_x_sc = data['test_x_sc']
            y_train = data['y_train']
            y_test = data['y_test']
            b_selected_features, accuracy = get_best_features_evo(algo, train_x_sc, y_train, test_x_sc, y_test)
            bal_data_res.append([file, b_selected_features, accuracy])

        imbal_data_res = []
        for file in os.listdir(imbal_dir):
            print("dataset: ",file)
            f_p = os.path.join(imbal_dir, file)
            data = get_data(f_p=f_p)
            train_x_sc = data['train_x_sc']
            test_x_sc = data['test_x_sc']
            y_train = data['y_train']
            y_test = data['y_test']
            b_selected_features, accuracy = get_best_features_evo(algo, train_x_sc, y_train, test_x_sc, y_test)
            imbal_data_res.append([file, b_selected_features, accuracy])


        res_data = pd.DataFrame([["Balanced Dataset", "",  ""]] + bal_data_res + [["Imbalanced Dataset", "",  ""]] + imbal_data_res)
        print(res_data)
        res_data.columns = ["Dataset", "Best Features", "Accuracy"]
        res_data.to_csv("DEAP_TRAD_FEAT_SEL/res_pop_5/above_15_res/"+ev_k+".csv", index=None)
    except Exception as e:
        print("error: ",e)


2025/02/13 11:55:35 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: Solving single objective optimization problem.


dataset:  processed_Autism Screening Adult.csv
dataset:  dim_above_15/balanced\processed_Autism Screening Adult.csv
Algo:  OriginalFFA(epoch=10, pop_size=5, gamma=0.001, beta_base=2.0, alpha=0.2, alpha_damp=0.99, delta=0.05, exponent=2)


2025/02/13 11:55:37 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 1, Current best: 0.008, Global best: 0.008, Runtime: 1.70303 seconds
2025/02/13 11:55:39 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 2, Current best: 0.008, Global best: 0.008, Runtime: 1.65448 seconds
2025/02/13 11:55:40 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 3, Current best: 0.005, Global best: 0.005, Runtime: 1.89331 seconds
2025/02/13 11:55:42 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 4, Current best: 0.005, Global best: 0.005, Runtime: 1.78516 seconds
2025/02/13 11:55:44 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 5, Current best: 0.005, Global best: 0.005, Runtime: 1.62616 seconds
2025/02/13 11:55:45 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 6, Current best: 0.005, Global best: 0.005, Runtime: 1.46661 seconds
2025/02/13 11:55:47 AM, INFO, mealpy.swarm_based.FFA.Origi

b_selected_features:  [ 0  4 10 14 17]
accuracy:  1.0
dataset:  processed_Chess (King-Rook vs. King-Pawn).csv
dataset:  dim_above_15/balanced\processed_Chess (King-Rook vs. King-Pawn).csv
Algo:  OriginalFFA(epoch=10, pop_size=5, gamma=0.001, beta_base=2.0, alpha=0.2, alpha_damp=0.99, delta=0.05, exponent=2)


2025/02/13 11:55:52 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: Solving single objective optimization problem.
2025/02/13 11:55:55 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 1, Current best: 0.06909489051094896, Global best: 0.06909489051094896, Runtime: 2.89365 seconds
2025/02/13 11:55:58 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 2, Current best: 0.06909489051094896, Global best: 0.06909489051094896, Runtime: 2.70858 seconds
2025/02/13 11:56:01 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 3, Current best: 0.06909489051094896, Global best: 0.06909489051094896, Runtime: 2.74084 seconds
2025/02/13 11:56:03 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 4, Current best: 0.06909489051094896, Global best: 0.06909489051094896, Runtime: 2.70607 seconds
2025/02/13 11:56:06 AM, INFO, mealpy.swarm_based.FFA.OriginalFFA: >>>Problem: P, Epoch: 5, Current best: 0.06909489051094896, Global best: 0.069094

...

b_selected_features:  [ 4  6  8 12 13 14 18]
accuracy:  1.0
dataset:  processed_Dermatology.csv
dataset:  dim_above_15/imbalanced\processed_Dermatology.csv
Algo:  DevSSA(epoch=10, pop_size=5, ST=0.8, PD=0.2, SD=0.1)


2025/02/13 12:14:04 PM, INFO, mealpy.swarm_based.SSA.DevSSA: Solving single objective optimization problem.
2025/02/13 12:14:06 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 1, Current best: 0.03675247524752477, Global best: 0.03675247524752477, Runtime: 1.17878 seconds
2025/02/13 12:14:08 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 2, Current best: 0.03675247524752477, Global best: 0.03675247524752477, Runtime: 1.32259 seconds
2025/02/13 12:14:09 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 3, Current best: 0.03675247524752477, Global best: 0.03675247524752477, Runtime: 1.10014 seconds
2025/02/13 12:14:10 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 4, Current best: 0.03675247524752477, Global best: 0.03675247524752477, Runtime: 1.20883 seconds
2025/02/13 12:14:11 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 5, Current best: 0.03675247524752477, Global best: 0.03675247524752477, Runtime: 1.34860 

b_selected_features:  [ 3  4 12 14 15 18 19 24 26 28 29 31]
accuracy:  0.9752475247524752
dataset:  processed_Thoracic Surgery.csv
dataset:  dim_above_15/imbalanced\processed_Thoracic Surgery.csv
Algo:  DevSSA(epoch=10, pop_size=5, ST=0.8, PD=0.2, SD=0.1)


2025/02/13 12:14:18 PM, INFO, mealpy.swarm_based.SSA.DevSSA: Solving single objective optimization problem.
2025/02/13 12:14:20 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 1, Current best: 0.22350000000000003, Global best: 0.22350000000000003, Runtime: 1.46418 seconds
2025/02/13 12:14:22 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 2, Current best: 0.22350000000000003, Global best: 0.22350000000000003, Runtime: 1.51165 seconds
2025/02/13 12:14:23 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 3, Current best: 0.22350000000000003, Global best: 0.22350000000000003, Runtime: 1.45415 seconds
2025/02/13 12:14:25 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 4, Current best: 0.16949999999999998, Global best: 0.16949999999999998, Runtime: 1.49893 seconds
2025/02/13 12:14:26 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 5, Current best: 0.16949999999999998, Global best: 0.16949999999999998, Runtime: 1.57469 

b_selected_features:  [ 1  2  3  5 11 14 15]
accuracy:  0.8375
                                                    0  \
0                                    Balanced Dataset   
1                processed_Autism Screening Adult.csv   
2       processed_Chess (King-Rook vs. King-Pawn).csv   
3                        processed_Cylinder Bands.csv   
4         processed_Diabetic Retinopathy Debrecen.csv   
5             processed_Early Stage Diabetes Risk.csv   
6   processed_Estimation of obesity levels based o...   
7                             processed_Hepatitis.csv   
8              processed_Hepatocellular Carcinoma.csv   
9                         processed_Primary Tumor.csv   
10                                 Imbalanced Dataset   
11             processed_Audiology (Standardized).csv   
12                     processed_Cardiotocography.csv   
13                      processed_Cervical Cancer.csv   
14               processed_Chronic Kidney Disease.csv   
15                       

In [10]:
bal_dir = "dim_under_15/balanced"
imbal_dir = "dim_under_15/imbalanced"

done_list = os.listdir("DEAP_TRAD_FEAT_SEL/res_pop_5/under_15_res/")

for ev_k in evo_algorithms:

    try:

        if (ev_k+".csv" in done_list):
            continue

        algo = evo_algorithms[ev_k]
        bal_data_res = []
        for file in os.listdir(bal_dir):
            print("dataset: ",file)
            f_p = os.path.join(bal_dir, file)
            data = get_data(f_p=f_p)
            train_x_sc = data['train_x_sc']
            test_x_sc = data['test_x_sc']
            y_train = data['y_train']
            y_test = data['y_test']
            b_selected_features, accuracy = get_best_features_evo(algo, train_x_sc, y_train, test_x_sc, y_test)
            bal_data_res.append([file, b_selected_features, accuracy])

        imbal_data_res = []
        for file in os.listdir(imbal_dir):
            print("dataset: ",file)
            f_p = os.path.join(imbal_dir, file)
            data = get_data(f_p=f_p)
            train_x_sc = data['train_x_sc']
            test_x_sc = data['test_x_sc']
            y_train = data['y_train']
            y_test = data['y_test']
            b_selected_features, accuracy = get_best_features_evo(algo, train_x_sc, y_train, test_x_sc, y_test)
            imbal_data_res.append([file, b_selected_features, accuracy])


        res_data = pd.DataFrame([["Balanced Dataset", "",  ""]] + bal_data_res + [["Imbalanced Dataset", "",  ""]] + imbal_data_res)
        print(res_data)
        res_data.columns = ["Dataset", "Best Features", "Accuracy"]
        res_data.to_csv("DEAP_TRAD_FEAT_SEL/res_pop_5/under_15_res/"+ev_k+".csv", index=None)
    except Exception as e:
        print("error: ",e)

2025/02/13 12:14:34 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: Solving single objective optimization problem.


dataset:  processed_Dishonest Internet users.csv
dataset:  dim_under_15/balanced\processed_Dishonest Internet users.csv
Algo:  AIW_PSO(epoch=10, pop_size=5, c1=2.05, c2=2.05, alpha=0.4)


2025/02/13 12:14:35 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 1, Current best: 0.022618556701030967, Global best: 0.022618556701030967, Runtime: 0.53373 seconds
2025/02/13 12:14:36 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 2, Current best: 0.022618556701030967, Global best: 0.022618556701030967, Runtime: 0.53729 seconds
2025/02/13 12:14:37 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 3, Current best: 0.022618556701030967, Global best: 0.022618556701030967, Runtime: 0.57734 seconds
2025/02/13 12:14:37 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 4, Current best: 0.022618556701030967, Global best: 0.022618556701030967, Runtime: 0.52448 seconds
2025/02/13 12:14:38 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 5, Current best: 0.022618556701030967, Global best: 0.022618556701030967, Runtime: 0.57384 seconds
2025/02/13 12:14:38 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 6

b_selected_features:  [0 2 3]
accuracy:  1.0
dataset:  processed_Habermans Survival.csv
dataset:  dim_under_15/balanced\processed_Habermans Survival.csv
Algo:  AIW_PSO(epoch=10, pop_size=5, c1=2.05, c2=2.05, alpha=0.4)


2025/02/13 12:14:41 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: Solving single objective optimization problem.
2025/02/13 12:14:42 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 1, Current best: 0.2737391304347826, Global best: 0.2737391304347826, Runtime: 0.55808 seconds
2025/02/13 12:14:42 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 2, Current best: 0.2737391304347826, Global best: 0.2737391304347826, Runtime: 0.61357 seconds
2025/02/13 12:14:43 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 3, Current best: 0.2737391304347826, Global best: 0.2737391304347826, Runtime: 0.45233 seconds
2025/02/13 12:14:43 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 4, Current best: 0.2737391304347826, Global best: 0.2737391304347826, Runtime: 0.45086 seconds
2025/02/13 12:14:44 PM, INFO, mealpy.swarm_based.PSO.AIW_PSO: >>>Problem: P, Epoch: 5, Current best: 0.2737391304347826, Global best: 0.2737391304347826, Runtime: 0.63266 seco

...

b_selected_features:  [0]
accuracy:  0.8417508417508418
dataset:  processed_Fertility.csv
dataset:  dim_under_15/imbalanced\processed_Fertility.csv
Algo:  DevSSA(epoch=10, pop_size=5, ST=0.8, PD=0.2, SD=0.1)


2025/02/13 02:05:27 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 1, Current best: 0.04273584905660376, Global best: 0.04273584905660376, Runtime: 0.63082 seconds
2025/02/13 02:05:27 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 2, Current best: 0.04273584905660376, Global best: 0.04273584905660376, Runtime: 0.61823 seconds
2025/02/13 02:05:28 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 3, Current best: 0.04273584905660376, Global best: 0.04273584905660376, Runtime: 0.56892 seconds
2025/02/13 02:05:28 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 4, Current best: 0.04273584905660376, Global best: 0.04273584905660376, Runtime: 0.63743 seconds
2025/02/13 02:05:29 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 5, Current best: 0.021867924528301882, Global best: 0.021867924528301882, Runtime: 0.56074 seconds
2025/02/13 02:05:30 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 6, Current best

b_selected_features:  [1 4 6]
accuracy:  0.9811320754716981
dataset:  processed_Heart Disease.csv
dataset:  dim_under_15/imbalanced\processed_Heart Disease.csv
Algo:  DevSSA(epoch=10, pop_size=5, ST=0.8, PD=0.2, SD=0.1)


2025/02/13 02:05:33 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 1, Current best: 0.1685365853658537, Global best: 0.1685365853658537, Runtime: 0.82015 seconds
2025/02/13 02:05:34 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 2, Current best: 0.14921138211382112, Global best: 0.14921138211382112, Runtime: 0.98545 seconds
2025/02/13 02:05:35 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 3, Current best: 0.14721138211382112, Global best: 0.14721138211382112, Runtime: 1.09612 seconds
2025/02/13 02:05:37 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 4, Current best: 0.14714634146341465, Global best: 0.14714634146341465, Runtime: 1.09986 seconds
2025/02/13 02:05:38 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 5, Current best: 0.14714634146341465, Global best: 0.14714634146341465, Runtime: 1.07466 seconds
2025/02/13 02:05:39 PM, INFO, mealpy.swarm_based.SSA.DevSSA: >>>Problem: P, Epoch: 6, Current best: 0.

b_selected_features:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
accuracy:  0.8658536585365854
                                                    0  \
0                                    Balanced Dataset   
1              processed_Dishonest Internet users.csv   
2                    processed_Habermans Survival.csv   
3                            processed_Hayes Roth.csv   
4   processed_ILPD (Indian Liver Patient Dataset).csv   
5                       processed_Liver Disorders.csv   
6                                  Imbalanced Dataset   
7                               processed_Abalone.csv   
8             processed_Blood Transfusion Service.csv   
9                         processed_Breast Cancer.csv   
10     processed_Climate Model Simulation Crashes.csv   
11                            processed_Fertility.csv   
12                        processed_Heart Disease.csv   

                                             1         2  
0                                                